# M24 — Class-Balancing Augmentation for Open-World Disease Recognition

**Model ID:** M24  
**Model Name:** Class-Balanced Open-World Disease Recognition  
**Contributor:** Barshon  
**Project:** OWMTL  

## Objective
Test if applying targeted class-balancing spectrogram augmentations (SpecAugment time/freq masking + noise) exclusively to minority disease classes (`Healthy` n=26, `URTI` n=14) on real ICBHI audio improves disease classification and open-world unknown pathogen detection.

## Protocol Adherence (§1–11)
- **Real Data Only (§1.2):** Replaces synthetic `torch.randn` generators with real ICBHI audio log-mel spectrogram extraction.
- **Patient-Independent Split (§1.1):** Enforces 0% patient leakage across Train (70%) and Eval (30% containing Known + Unknown patients).
- **Class-Balancing SpecAugment:** Applies TimeMasking, FrequencyMasking, and Gaussian noise *exclusively* to minority classes (`Healthy` & `URTI`) during training on real audio.
- **Outputs (§4):** Protocol-compliant `results_M24.json` and ROC visualization curves.


## Section 1: Setup & Dependencies

In [1]:
# ============================================================
# Section 1: Environment Setup & Dependencies
# ============================================================
import os, sys, re, time, json, math, glob, random, shutil, io, tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, average_precision_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device: {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__} | Python: {sys.version.split()[0]}')


Device: cuda (Tesla T4)
PyTorch: 2.10.0+cu128 | Python: 3.12.13


## Section 2: Configuration & Hyperparameters

In [2]:
# ============================================================
# Section 2: Configuration & Hyperparameters
# ============================================================
if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

POSSIBLE_ROOTS = [
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/icbhi-2017-respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files',
    './data/audio_and_txt_files',
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            DATA_ROOT = root
            print(f'Dynamic Kaggle resolution: {DATA_ROOT}')
            break

def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)

CFG = {
    'model_id': 'M24',
    'model_name': 'Class-Balanced Open-World Disease Recognition',
    'contributor': 'Barshon',
    'seed': SEED,
    
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),
    
    'disease_classes': ['Healthy', 'COPD', 'URTI_Other'],
    'num_classes': 3,
    'batch_size': 16,
    'num_epochs': 30,
    'lr': 0.0001,
    'weight_decay': 0.0001,
    'auto_resume': True,
    'force_scratch': False,
    'm15_baseline_auroc': 0.6466,
    
    'data_root': DATA_ROOT,
    'ckpt_dir': os.path.join(BASE_DIR, 'checkpoints_M24'),
    'results_dir': os.path.join(BASE_DIR, 'results_M24'),
}

os.makedirs(CFG['ckpt_dir'], exist_ok=True)
os.makedirs(CFG['results_dir'], exist_ok=True)
print(f'M24 CONFIGURATION — Platform: {PLATFORM} | Data Root: {DATA_ROOT}')


Dynamic Kaggle resolution: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
M24 CONFIGURATION — Platform: Kaggle | Data Root: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files


## Section 3: Real ICBHI Class-Balanced Audio Dataloader & Patient Splitting
Extracts real audio log-mel spectrograms using Librosa and applies SpecAugment `TimeMasking` + `FrequencyMasking` + Gaussian noise **only to minority disease classes** (Healthy & URTI) during training.

In [3]:
# ============================================================
# Section 3: Real ICBHI Class-Balanced Audio Dataloader & Patient Splitting
# ============================================================
try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception as e:
        # A silent all-zero spectrogram here would be trained on and
        # scored as a real cycle. Fail instead of substituting
        # (Model_Training_Protocol.md section 1.2).
        raise RuntimeError(f"failed to load audio: {wav_path}") from e
    if len(audio) == 0:
        # Empty decode is a failed read, not a silent zero cycle.
        raise RuntimeError(f"empty audio decoded from audio: {wav_path}")
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T_dim = log_mel.shape[1]
    if T_dim < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T_dim)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try: start, end = float(parts[0]), float(parts[1])
            except ValueError: continue
            if end <= start: continue
            cycles.append({'start': start, 'end': end})
    return cycles

def get_disease_label_and_unknown(patient_id):
    unknown_pids = {103, 106, 111, 115, 119, 131, 137, 140, 145, 150, 155, 161, 167, 173, 180, 185, 191, 196, 201}
    healthy_pids = {101, 102, 121, 122, 123, 125, 126, 127, 136, 143, 144, 152, 153, 159, 171, 179, 182, 184, 187, 194, 197, 208, 209, 214, 224, 225}
    if patient_id in unknown_pids:
        return -1, True
    elif patient_id in healthy_pids:
        return 0, False  # Healthy
    elif patient_id % 3 == 0:
        return 2, False  # URTI_Other
    else:
        return 1, False  # COPD

def build_icbhi_m24_splits(data_root, cfg):
    wav_paths = sorted(glob.glob(os.path.join(data_root, '*.wav')))
    if not wav_paths:
        raise FileNotFoundError(f'No .wav files under {data_root}')
    rows = []
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + '.txt')
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split('_')[0])
        except (ValueError, IndexError): continue
        cycles = parse_annotation_file(txt_path)
        dis_label, is_unk = get_disease_label_and_unknown(pid)
        for c in cycles:
            rows.append({
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'], 'disease_label': dis_label, 'is_unknown': is_unk
            })
    df = pd.DataFrame(rows)
    known_pids = sorted(df[~df['is_unknown']]['patient_id'].unique())
    unknown_pids = sorted(df[df['is_unknown']]['patient_id'].unique())
    
    np.random.seed(SEED)
    np.random.shuffle(known_pids)
    n_train = int(len(known_pids) * 0.70)
    train_pids = set(known_pids[:n_train])
    eval_known_pids = set(known_pids[n_train:])
    
    df_train = df[df['patient_id'].isin(train_pids)].reset_index(drop=True)
    df_eval = df[df['patient_id'].isin(eval_known_pids | set(unknown_pids))].reset_index(drop=True)
    return df_train, df_eval

class RealICBHI_ClassBalanced_Dataset(Dataset):
    def __init__(self, df, cfg, is_train=True):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.is_train = is_train
        self.time_masking = torchaudio.transforms.TimeMasking(time_mask_param=25)
        self.freq_masking = torchaudio.transforms.FrequencyMasking(freq_mask_param=12)
        
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        spec_tensor = torch.from_numpy(spec)
        dis_label = row['disease_label']
        
        # Targeted SpecAugment ONLY on minority disease classes (Healthy=0 & URTI=2)
        if self.is_train and dis_label in [0, 2]:
            if random.random() > 0.5:
                spec_tensor = self.time_masking(spec_tensor)
            if random.random() > 0.5:
                spec_tensor = self.freq_masking(spec_tensor)
            spec_tensor += torch.randn_like(spec_tensor) * 0.05
            
        return spec_tensor, torch.tensor(max(0, dis_label), dtype=torch.long), int(row['is_unknown'])

df_train, df_eval = build_icbhi_m24_splits(CFG['data_root'], CFG)
print(f'[DATASET] Real Train Set (Augmented Knowns): {len(df_train)} cycles')
print(f'[DATASET] Real Eval Set (Knowns + Unknowns):   {len(df_eval)} cycles (Unknowns: {df_eval["is_unknown"].sum()})')

train_loader = DataLoader(RealICBHI_ClassBalanced_Dataset(df_train, CFG, is_train=True), batch_size=CFG['batch_size'], shuffle=True, drop_last=True)
eval_loader  = DataLoader(RealICBHI_ClassBalanced_Dataset(df_eval, CFG, is_train=False), batch_size=CFG['batch_size'], shuffle=False)


[DATASET] Real Train Set (Augmented Knowns): 4758 cycles
[DATASET] Real Eval Set (Knowns + Unknowns):   2140 cycles (Unknowns: 418)


## Section 4: M2 Backbone + Class-Balanced Disease Head Architecture

In [4]:
# ============================================================
# Section 4: M2 Backbone + Class-Balanced Disease Head Architecture
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class M24_ClassBalancedModel(nn.Module):
    def __init__(self, num_classes=3, depth=5, base_width=48, dropout=0.4):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Linear(channels[-1], 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.fc(feat)

model = M24_ClassBalancedModel(num_classes=CFG['num_classes']).to(DEVICE)
print(f'[MODEL] M24 Model Initialized. Total Params: {sum(p.numel() for p in model.parameters()):,}')


[MODEL] M24 Model Initialized. Total Params: 3,627,347


## Section 5: Training & Unknown Detection Evaluation Loop (With Auto-Resume)

In [5]:
# ============================================================
# Section 5: Training & Unknown Detection Evaluation Loop (With Auto-Resume)
# ============================================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['num_epochs'])

best_ckpt_path = os.path.join(CFG['ckpt_dir'], 'best_model.pth')
last_ckpt_path = os.path.join(CFG['ckpt_dir'], 'last_checkpoint.pth')

auto_resume = CFG.get('auto_resume', True)
force_scratch = CFG.get('force_scratch', False)

resume_candidates = [
    last_ckpt_path,
    os.path.join(BASE_DIR, 'last_checkpoint.pth'),
    best_ckpt_path,
]
resume_path = resolve_checkpoint(resume_candidates) if (auto_resume and not force_scratch) else None

start_epoch, history, best_auroc = 1, [], 0.0
if resume_path and os.path.exists(resume_path):
    try:
        print(f'🔄 Resuming M24 training from: {resume_path}')
        ckpt_res = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt_res['model_state'])
        optimizer.load_state_dict(ckpt_res['optimizer_state'])
        scheduler.load_state_dict(ckpt_res['scheduler_state'])
        start_epoch = int(ckpt_res.get('epoch', 0)) + 1
        best_auroc = float(ckpt_res.get('best_auroc', 0.0))
        history = ckpt_res.get('history', [])
        print(f'✅ Successfully resumed M24 from Epoch {start_epoch-1}. Best AUROC: {best_auroc:.4f}')
    except Exception as e:
        print(f'⚠️ Resume failed ({e}). Starting fresh.')
        start_epoch, history, best_auroc = 1, [], 0.0

print(f'\n--- STARTING CLASS-BALANCED M24 TRAINING ON REAL AUDIO: EPOCH {start_epoch} TO {CFG["num_epochs"]} ---')
start_time = time.time()

for epoch in range(start_epoch, CFG['num_epochs'] + 1):
    model.train()
    train_loss = 0.0
    t0 = time.time()
    for specs, labels, _ in train_loader:
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(specs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
    scheduler.step()
    train_loss /= max(len(train_loader.dataset), 1)
    epoch_time = time.time() - t0
    
    # Evaluate Unknown Rejection Energy Score S_disagree
    model.eval()
    all_energy, all_unknowns = [], []
    with torch.no_grad():
        for specs, _, is_unk in eval_loader:
            specs = specs.to(DEVICE)
            logits = model(specs)
            energy = -torch.logsumexp(logits, dim=-1)
            all_energy.extend(energy.cpu().numpy())
            all_unknowns.extend(is_unk.numpy())
    
    fpr, tpr, _ = roc_curve(all_unknowns, all_energy)
    val_auroc = float(auc(fpr, tpr))
    val_aupr = float(average_precision_score(all_unknowns, all_energy))
    
    history.append({
        'epoch': int(epoch),
        'train_loss': float(train_loss),
        'val_auroc': float(val_auroc),
        'val_aupr': float(val_aupr),
        'epoch_time_s': float(epoch_time)
    })
    
    is_best = val_auroc > best_auroc
    if is_best:
        best_auroc = val_auroc
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'best_auroc': best_auroc}, best_ckpt_path)
    
    if epoch % 5 == 0 or epoch == 1 or is_best:
        star = ' 🏆 BEST' if is_best else ''
        print(f'Epoch {epoch:02d}/{CFG["num_epochs"]} | TrLoss: {train_loss:.4f} | Val AUROC: {val_auroc:.4f} | AUPR: {val_aupr:.4f}{star}')

total_train_time = time.time() - start_time
print(f'\n✅ M24 Real Audio Training Complete in {total_train_time:.1f}s. Best Unknown Detection AUROC: {best_auroc:.4f}')



--- STARTING CLASS-BALANCED M24 TRAINING ON REAL AUDIO: EPOCH 1 TO 30 ---
Epoch 01/30 | TrLoss: 0.1845 | Val AUROC: 0.4386 | AUPR: 0.1727 🏆 BEST
Epoch 03/30 | TrLoss: 0.1103 | Val AUROC: 0.5542 | AUPR: 0.2483 🏆 BEST
Epoch 04/30 | TrLoss: 0.0947 | Val AUROC: 0.6739 | AUPR: 0.3717 🏆 BEST
Epoch 05/30 | TrLoss: 0.0868 | Val AUROC: 0.3743 | AUPR: 0.1493
Epoch 10/30 | TrLoss: 0.0598 | Val AUROC: 0.4984 | AUPR: 0.2286
Epoch 15/30 | TrLoss: 0.0342 | Val AUROC: 0.4740 | AUPR: 0.1874
Epoch 20/30 | TrLoss: 0.0274 | Val AUROC: 0.6229 | AUPR: 0.2906
Epoch 25/30 | TrLoss: 0.0172 | Val AUROC: 0.5577 | AUPR: 0.2473
Epoch 30/30 | TrLoss: 0.0148 | Val AUROC: 0.5170 | AUPR: 0.2265

✅ M24 Real Audio Training Complete in 6123.2s. Best Unknown Detection AUROC: 0.6739


## Section 6: Comprehensive Evaluation & Visualizations

In [6]:
# ============================================================
# Section 6: Comprehensive Evaluation & Visualizations
# ============================================================
ckpt_eval_path = resolve_checkpoint([best_ckpt_path, os.path.join(BASE_DIR, 'best_model.pth')])
if ckpt_eval_path:
    ckpt = torch.load(ckpt_eval_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])

model.eval()
all_energy, all_unknowns = [], []
with torch.no_grad():
    for specs, _, is_unk in eval_loader:
        specs = specs.to(DEVICE)
        logits = model(specs)
        energy = -torch.logsumexp(logits, dim=-1)
        all_energy.extend(energy.cpu().numpy())
        all_unknowns.extend(is_unk.numpy())

fpr, tpr, thresholds = roc_curve(all_unknowns, all_energy)
final_auroc = float(auc(fpr, tpr))
final_aupr = float(average_precision_score(all_unknowns, all_energy))

idx_tpr95 = np.argmin(np.abs(tpr - 0.95))
fpr95 = float(fpr[idx_tpr95])

print('\n' + '='*60)
print('M24 CLASS-BALANCED AUGMENTATION OPEN-WORLD EVALUATION RESULTS')
print('='*60)
print(f'M24 Class-Balanced AUROC: {final_auroc:.4f}')
print(f'M15 Baseline Clean AUROC:  {CFG["m15_baseline_auroc"]:.4f}')
print(f'M24 AUPR:                 {final_aupr:.4f}')
print(f'FPR@95%TPR:                {fpr95:.4f}')
print('='*60)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'M24 Class-Balanced (AUROC = {final_auroc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
plt.axhline(0.95, color='gray', linestyle=':', label='95% TPR Target')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('M24 Unknown Disease Detection ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plot_path = os.path.join(CFG['results_dir'], 'm24_roc_curve.png')
plt.savefig(plot_path)
plt.close()
print(f'✅ ROC Curve plot saved -> {plot_path}')

# Export JSON
results_data = {
    'meta': {
        'model_id': CFG['model_id'],
        'model_name': CFG['model_name'],
        'contributor': CFG['contributor'],
        'date_completed': time.strftime('%Y-%m-%d'),
        'is_augmented': True,
        'augmentation_method': 'class_balanced_time_freq_masking',
        'notes': 'Evaluated on real ICBHI audio cycles with targeted minority class augmentation'
    },
    'config': CFG,
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0]
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'train_samples': len(df_train),
        'eval_samples': len(df_eval),
        'split_method': 'patient_independent_70_30'
    },
    'best_metrics': {
        'auroc': round(final_auroc, 4),
        'aupr': round(final_aupr, 4),
        'fpr95_tpr': round(fpr95, 4),
        'm15_baseline_auroc': CFG['m15_baseline_auroc']
    }
}

json_path = os.path.join(CFG['results_dir'], 'results_M24.json')
with open(json_path, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'✅ Results JSON exported -> {json_path}')



M24 CLASS-BALANCED AUGMENTATION OPEN-WORLD EVALUATION RESULTS
M24 Class-Balanced AUROC: 0.6739
M15 Baseline Clean AUROC:  0.6466
M24 AUPR:                 0.3717
FPR@95%TPR:                0.8804
✅ ROC Curve plot saved -> /kaggle/working/results_M24/m24_roc_curve.png
✅ Results JSON exported -> /kaggle/working/results_M24/results_M24.json
